# Phase 3 (A) — Ising rồi QAOA trên mô phỏng nhỏ

Chạy lần lượt trong VS Code với kernel Conda `nt409-khiem` (Python 3.12). Phase 2 đã cung cấp QUBO; Phase 3 đổi cùng hàm năng lượng sang Ising rồi thử QAOA. **Đây là notebook thí điểm, chưa phải benchmark đầy đủ hoặc bằng chứng bài toán có nghiệm.**

Bốn fixture Phase 1/2 có 18–23 qubit. Notebook kiểm tra Ising trên chúng, nhưng chỉ chạy QAOA trên `p3_demo` (12 qubit, cùng schema và `build_qubo`). Bản này không lưu sẵn output: kết quả xuất hiện sau khi bạn chạy là kết quả của chính kernel đang chọn.

## Bước 1 — Kernel và đường dẫn dự án

Đưa repository vào `sys.path`, in phiên bản Qiskit. Nếu thiếu thư viện, hãy dùng đúng kernel Conda từ Phase 2. `ROOT` phải là thư mục chứa `src`, `data`, `notebooks`.

In [1]:
import json
import sys
import platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy
import qiskit
import qiskit_optimization

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "qubo.py").is_file() and (p / "data").is_dir()), None)
assert ROOT is not None, "Mở notebook trong repository nt409-task-placement."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.qubo import build_qubo, exact_qubo_reference, decode_qubo
from src.placement import evaluate_placement
from src.ising import build_ising, ising_energy, decode_samples, integer_to_bits

required = [ROOT / "src" / "ising.py",
            ROOT / "data" / "instances" / "p3_demo.json",
            ROOT / "data" / "reference_results" / "phase2_qubo.json"]
assert all(p.is_file() for p in required), [str(p) for p in required if not p.is_file()]
print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Qiskit Optimization:", qiskit_optimization.__version__)
print("Project:", ROOT)

Python: 3.12.14
Qiskit: 2.5.2
Qiskit Optimization: 0.7.0
Project: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement


## Bước 2 — Dữ liệu và số qubit

Bốn fixture cũ dùng để kiểm tra phép đổi QUBO → Ising. Demo mới có 2 task (mỗi task 1 CPU, 1 RAM): H0 chứa được cả hai, H1 chứa một. Nghiệm gốc tốt nhất cần **1 host**. Vì dung lượng nhỏ, demo chỉ dùng 12 qubit.

In [2]:
names = ["p0_reference", "p1_choice", "p1_tight", "p1_infeasible", "p3_demo"]
instances = {
    name: json.loads((ROOT / "data" / "instances" / f"{name}.json").read_text(encoding="utf-8"))
    for name in names
}
bundles = {name: build_qubo(instance) for name, instance in instances.items()}
rows = []
for name, bundle in bundles.items():
    rows.append({"instance": name, "x+y": bundle["base_count"],
                 "slack": bundle["qubo"].get_num_vars() - bundle["base_count"],
                 "qubits": bundle["qubo"].get_num_vars(), "P": bundle["penalty"]})
display(pd.DataFrame(rows))
assert bundles["p3_demo"]["qubo"].get_num_vars() == 12

,instance,x+y,slack,qubits,P
0,p0_reference,10,13,23,3.0
1,p1_choice,10,10,20,3.0
2,p1_tight,10,8,18,3.0
3,p1_infeasible,10,10,20,3.0
4,p3_demo,6,6,12,3.0


## Bước 3 — QUBO → Ising: $b=(1-Z)/2$

Một bit $b_k=0$ tương ứng eigenvalue $Z_k=+1$; $b_k=1$ tương ứng $Z_k=-1$. Thay vào QUBO tạo cost Hamiltonian gồm I, Z, ZZ. Qiskit trả về `(operator, offset)` với $E_{\mathrm{QUBO}}(b)=\langle b|H|b\rangle+\mathrm{offset}$. `offset` thay đổi mốc năng lượng, không đổi bit tối ưu.

In [3]:
ising = {name: build_ising(bundle) for name, bundle in bundles.items()}
for name, (operator, offset) in ising.items():
    print(f"{name:16s}: {operator.num_qubits:2d} qubit, {len(operator):3d} Pauli terms, offset={offset:g}")
    for label in operator.paulis.to_labels():
        assert set(label) <= {"I", "Z"} and label.count("Z") <= 2

p0_reference    : 23 qubit, 127 Pauli terms, offset=742
p1_choice       : 20 qubit, 102 Pauli terms, offset=175
p1_tight        : 18 qubit,  86 Pauli terms, offset=187
p1_infeasible   : 20 qubit, 102 Pauli terms, offset=319
p3_demo         : 12 qubit,  40 Pauli terms, offset=34


## Bước 4 — Kiểm tra Ising chấm cùng điểm với QUBO

Mỗi fixture: thử chuỗi toàn 0, từng bit đơn lẻ, từng cặp bit, và một nghiệm từ báo cáo Phase 2. Hai biểu thức đều là hàm bậc tối đa 2 trên bit, nên 0/1/2 bit bật đủ kiểm tra mọi hệ số của hàm. `ising_energy` đọc `Z` trên bit ở thứ tự qubit Qiskit; không cấp phát statevector (2^{23}) phần tử.

In [4]:
from itertools import combinations

phase2 = json.loads((ROOT / "data" / "reference_results" / "phase2_qubo.json").read_text(encoding="utf-8"))
reference_bits = {row["instance_id"]: row["example_minimizer"]["bits"] for row in phase2["results"]}
checked = {}
for name, bundle in bundles.items():
    operator, offset = ising[name]
    n = operator.num_qubits
    bitsets = [np.zeros(n, dtype=int)]
    for k in (1, 2):
        for indices in combinations(range(n), k):
            bits = np.zeros(n, dtype=int)
            bits[list(indices)] = 1
            bitsets.append(bits)
    if name in reference_bits:
        bitsets.append(np.array(reference_bits[name], dtype=int))
    for bits in bitsets:
        expected = float(bundle["qubo"].objective.evaluate(bits))
        observed = ising_energy(operator, offset, bits)
        assert np.isclose(expected, observed, rtol=0, atol=1e-7), (name, bits)
    checked[name] = len(bitsets)
print("PASS — QUBO energy = Ising energy + offset:", checked)

PASS — QUBO energy = Ising energy + offset: {'p0_reference': 278, 'p1_choice': 212, 'p1_tight': 173, 'p1_infeasible': 212, 'p3_demo': 79}


## Bước 5 — Hai đáp án chuẩn và mốc lấy mẫu đều

`exact_qubo_reference` tìm cực tiểu QUBO. Duyệt placement bằng bộ đánh giá Phase 1 xác định riêng số host tối ưu. Hai đại lượng trùng bằng 1 ở demo này, nhưng phải giữ ý nghĩa riêng.

Để đánh giá QAOA, tính thêm xác suất khi lấy ngẫu nhiên đều mọi chuỗi 12 bit. Đây là mốc cho **cùng QUBO**; nó không thay thế baseline heuristic/MILP của B. Chỉ duyệt 4096 chuỗi của demo, không duyệt toàn bộ chuỗi của fixture lớn.

In [5]:
from itertools import product

demo = bundles["p3_demo"]
truth = exact_qubo_reference(demo)
placement_results = [evaluate_placement(demo["instance"], list(p))
                     for p in product(range(demo["m"]), repeat=demo["n"])]
exact_best_objective = min(row["objective"] for row in placement_results if row["feasible"])
assert truth["minimum_energy"] == 1 and exact_best_objective == 1
assert all(item["decoded"]["feasible"] for item in truth["minimizers"])

num_bits = demo["qubo"].get_num_vars()
assert num_bits <= 12, "Chỉ duyệt toàn bộ bit cho demo nhỏ."
state_count = 2**num_bits
uniform_counts = {"feasible": 0, "optimal_placement": 0, "ground_state": 0}
uniform_energy_sum = 0.0
for index in range(state_count):
    bits = integer_to_bits(index, num_bits)
    decoded = decode_qubo(demo, bits)
    energy = float(demo["qubo"].objective.evaluate(bits))
    uniform_energy_sum += energy
    uniform_counts["feasible"] += decoded["feasible"]
    uniform_counts["optimal_placement"] += (decoded["feasible"] and decoded["objective"] == exact_best_objective)
    uniform_counts["ground_state"] += bool(np.isclose(energy, truth["minimum_energy"], rtol=0, atol=1e-7))
uniform_baseline = {key + "_probability": count / state_count for key, count in uniform_counts.items()}
uniform_baseline["mean_qubo_energy"] = uniform_energy_sum / state_count
print("Exact minimum QUBO energy:", truth["minimum_energy"])
print("Exact best placement objective:", exact_best_objective)
print("Uniform sampling baseline:", uniform_baseline)

# Independent check using Qiskit's actual basis-state expectation (12 qubits only).
from qiskit.quantum_info import Statevector
operator_demo, offset_demo = ising["p3_demo"]
known_bits = truth["minimizers"][0]["bits"]
known_index = sum(int(bit) << k for k, bit in enumerate(known_bits))
for index in [1, 1 << (num_bits - 1), known_index]:
    bits = integer_to_bits(index, num_bits)
    state = Statevector.from_int(index, 2**num_bits)
    observed = float(np.real(state.expectation_value(operator_demo))) + offset_demo
    expected = float(demo["qubo"].objective.evaluate(bits))
    assert np.isclose(observed, expected, rtol=0, atol=1e-7)
print("PASS — bit order matches Qiskit basis states.")


Exact minimum QUBO energy: 1.0
Exact best placement objective: 1
Uniform sampling baseline: {'feasible_probability': 0.0625, 'optimal_placement_probability': 0.015625, 'ground_state_probability': 0.000244140625, 'mean_qubo_energy': 34.0}
PASS — bit order matches Qiskit basis states.


## Bước 6 — QAOA $p=1$ trên mô phỏng

Cost layer là $e^{-i\gamma H_C}$, mixer layer là $e^{-i\beta H_M}$. COBYLA điều chỉnh góc dựa trên năng lượng kỳ vọng được ước lượng từ shots. Với COBYLA, `maxiter` giới hạn **số lần đánh giá hàm mục tiêu**; đạt ngân sách này không chứng minh hội tụ.

Mọi thiết lập nằm trong `CONFIG`, rồi được dùng lại khi lưu báo cáo. Có 1024 shots cho mỗi lần đánh giá; sau tối ưu thư viện còn lấy một batch mẫu cuối. Hai phép ước lượng có thể khác nhau do lấy mẫu. Callback lưu lịch sử đánh giá để đọc tiến trình.

In [6]:
from time import perf_counter
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA

CONFIG = {"seed": 42, "shots": 1024, "reps": 1,
          "maxiter": 35, "initial_point": [0.3, 0.7]}
assert len(CONFIG["initial_point"]) == 2 * CONFIG["reps"]
operator, offset = ising["p3_demo"]
history = []
def record_evaluation(eval_count, parameters, value, metadata):
    history.append({"evaluation": int(eval_count),
                    "parameters": np.asarray(parameters).tolist(),
                    "estimated_qubo_energy": float(np.real(value)) + offset})

sampler = StatevectorSampler(default_shots=CONFIG["shots"], seed=CONFIG["seed"])
algorithm = QAOA(sampler=sampler,
                 optimizer=COBYLA(maxiter=CONFIG["maxiter"]),
                 reps=CONFIG["reps"], initial_point=CONFIG["initial_point"],
                 callback=record_evaluation)
started_at = datetime.now(timezone.utc).isoformat()
started = perf_counter()
result = algorithm.compute_minimum_eigenvalue(operator)
elapsed_seconds = perf_counter() - started
print("Optimizer's estimated QUBO expectation:", float(np.real(result.eigenvalue)) + offset)
print("Objective evaluations:", result.cost_function_evals, "/", CONFIG["maxiter"])
print("Elapsed seconds:", round(elapsed_seconds, 2))
print("Observed final bitstrings:", len(result.eigenstate))

Optimizer's estimated QUBO expectation: 14.3759765625
Objective evaluations: 34 / 35
Elapsed seconds: 1.04
Observed final bitstrings: 774


## Bước 7 — Đánh giá batch mẫu cuối

Qubit 0 nằm bên phải trong cách in bitstring của Qiskit. Helper đổi mẫu về đúng thứ tự biến QUBO, rồi kiểm tra mọi mẫu bằng bộ đánh giá Phase 1.

- `feasible_probability`: placement x,y hợp lệ, có thể còn bit slack sai.
- `optimal_placement_probability`: placement hợp lệ có số host tối ưu theo Phase 1.
- `qubo_ground_state_probability`: toàn bộ bit, gồm slack, đạt cực tiểu QUBO.

Bảng đối chiếu thêm lấy mẫu đều ở Bước 5. Kỳ vọng từ batch cuối được tính lại từ đúng phân phối đang đánh giá; không giả định bằng ước lượng mà optimizer đã dùng. Một mẫu tối ưu đơn lẻ hoặc một seed chưa đủ kết luận QAOA tốt hơn baseline.

In [7]:
samples = decode_samples(demo, result.eigenstate, operator, offset)
assert samples and all(s["probability"] >= 0 for s in samples)
assert np.isclose(sum(s["probability"] for s in samples), 1, rtol=0, atol=1e-6)
feasible = [s for s in samples if s["decoded"]["feasible"]]
valid_probability = sum(s["probability"] for s in feasible)
optimal_placement_probability = sum(s["probability"] for s in feasible
                                    if s["decoded"]["objective"] == exact_best_objective)
ground_state_probability = sum(s["probability"] for s in samples
                               if np.isclose(s["energy"], truth["minimum_energy"], rtol=0, atol=1e-7))
final_sample_mean_energy = sum(s["probability"] * s["energy"] for s in samples)
best_feasible = min(feasible,
                    key=lambda s: (s["decoded"]["objective"], s["energy"], -s["probability"])) if feasible else None
comparison = pd.DataFrame([
    {"method": "Uniform sampling", "feasible": uniform_baseline["feasible_probability"],
     "optimal placement": uniform_baseline["optimal_placement_probability"],
     "QUBO ground state": uniform_baseline["ground_state_probability"],
     "mean QUBO E": uniform_baseline["mean_qubo_energy"]},
    {"method": "QAOA final samples", "feasible": valid_probability,
     "optimal placement": optimal_placement_probability, "QUBO ground state": ground_state_probability,
     "mean QUBO E": final_sample_mean_energy},
])
display(comparison)
print("Most frequent valid?:", samples[0]["decoded"]["feasible"])
print("Best observed valid placement:", None if best_feasible is None else best_feasible["decoded"])
display(pd.DataFrame([{"probability": s["probability"], "QUBO E": s["energy"],
                       "valid": s["decoded"]["feasible"], "hosts": s["decoded"]["objective"],
                       "x": s["decoded"]["x"]} for s in samples[:10]]))

,method,feasible,optimal placement,QUBO ground state,mean QUBO E
0,Uniform sampling,0.062500,0.015625,0.000244,34.000000
1,QAOA final samples,0.144531,0.024414,0.001953,14.375977


Most frequent valid?: False
Best observed valid placement: {'feasible': True, 'objective': 1, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[1, 0], [1, 0]], 'y': [1, 0], 'usage': [[2, 2], [0, 0]]}


,probability,QUBO E,valid,hosts,x
0,0.004883,8.0,False,NaN,"[[0, 0], [0, 1]]"
1,0.003906,8.0,False,NaN,"[[0, 0], [1, 0]]"
2,0.003906,8.0,False,NaN,"[[1, 0], [0, 0]]"
3,0.003906,8.0,False,NaN,"[[1, 0], [0, 0]]"
4,0.003906,13.0,False,NaN,"[[0, 1], [0, 0]]"
5,0.003906,8.0,True,2.0,"[[1, 0], [0, 1]]"
6,0.003906,2.0,True,2.0,"[[0, 1], [1, 0]]"
7,0.002930,5.0,True,2.0,"[[1, 0], [0, 1]]"
8,0.002930,13.0,False,NaN,"[[0, 0], [1, 0]]"
9,0.002930,10.0,False,NaN,"[[0, 0], [0, 1]]"


## Bước 8 — Lưu báo cáo có thể kiểm tra lại

Lưu cấu hình thực tế, phiên bản, thời gian, đáp án chuẩn, mốc lấy mẫu đều, lịch sử tối ưu và mọi mẫu ở batch cuối. C có thể tính lại các tỷ lệ từ `final_samples` mà không cần chạy QAOA lần nữa.

Đây là báo cáo một lần chạy thử. Nếu không đo được placement hợp lệ, báo cáo vẫn ghi `best_observed_valid = None`; không đổi kết quả để khớp đáp án chuẩn.

In [8]:
report = {
    "schema_version": 2,
    "method": "QAOA_StatevectorSampler_COBYLA",
    "scope": "single_seed_12_qubit_pilot",
    "instance_id": "p3_demo",
    "started_at_utc": started_at,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "numpy_version": np.__version__, "scipy_version": scipy.__version__,
    "qiskit_version": qiskit.__version__,
    "qiskit_optimization_version": qiskit_optimization.__version__,
    "num_qubits": operator.num_qubits,
    "config": CONFIG.copy(),
    "cost_function_evals": int(result.cost_function_evals),
    "elapsed_seconds": elapsed_seconds,
    "optimal_point": np.asarray(result.optimal_point).tolist(),
    "parameter_names": [str(p) for p in algorithm.ansatz.parameters],
    "ising_offset": offset,
    "exact_min_energy": truth["minimum_energy"],
    "exact_best_objective": exact_best_objective,
    "optimizer_estimated_qubo_expectation": float(np.real(result.eigenvalue)) + offset,
    "final_sample_mean_qubo_energy": final_sample_mean_energy,
    "feasible_probability": valid_probability,
    "optimal_placement_probability": optimal_placement_probability,
    "qubo_ground_state_probability": ground_state_probability,
    "uniform_baseline": uniform_baseline,
    "most_frequent_valid": samples[0]["decoded"]["feasible"],
    "best_observed_valid": None if best_feasible is None else {
        "bits": best_feasible["bits"], "energy": best_feasible["energy"],
        "objective": best_feasible["decoded"]["objective"],
        "x": best_feasible["decoded"]["x"], "y": best_feasible["decoded"]["y"],
    },
    "optimization_history": history,
    "final_samples": [{"bits": s["bits"], "probability": s["probability"],
                       "energy": s["energy"], "feasible": s["decoded"]["feasible"],
                       "objective": s["decoded"]["objective"]} for s in samples],
}
path = ROOT / "data" / "reference_results" / "phase3_ising_qaoa.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Saved:", path)
print("Pilot run recorded. Four large fixtures were checked for Ising only.")

Saved: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase3_ising_qaoa.json
Pilot run recorded. Four large fixtures were checked for Ising only.


## Sau notebook

- A giữ lại notebook/output và báo cáo JSON. Nếu QAOA không đo được nghiệm hợp lệ, không sửa kết quả; tăng `maxiter`/shots hoặc thử seed khác ở lượt thí nghiệm riêng.
- B dùng cùng JSON `p3_demo` và các fixture cũ cho heuristic/MILP.
- C phân biệt số host của placement hợp lệ, năng lượng QUBO và kỳ vọng Ising (+ offset).
- QAOA trên 18–23 qubit thuộc lượt mở rộng có kiểm soát tài nguyên; không suy ra khả năng chạy máy lượng tử thật từ mô phỏng này.